#### 02 — Model Training & Selection

Trains and compares models for **Model A (intent/category)** and **Model B (priority)**, including hyperparameter tuning, error analysis and engineered-features experiment that shaped final production configuration.

In [1]:
import sys

sys.path.insert(0, '..')

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

RANDOM_SEED = 42

df = pd.read_csv("../data/clean_tickets.csv")
# Guard against CSV round-trip bug
df['subject'] = df['subject'].fillna('')
print(f"Loaded {len(df)} rows")
df['category'].value_counts()

Loaded 11691 rows


category
Technical Support    5238
General Inquiry      4826
Billing              1298
Sales                 329
Name: count, dtype: int64

#### Part A — Model A: Intent Classification

##### A.1 Baseline: naive TF-IDF config, 6-class taxonomy

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['category'], test_size=0.2, random_state=RANDOM_SEED, stratify=df['category']
)

baseline_vec = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=2, stop_words='english', sublinear_tf=True)
Xtr = baseline_vec.fit_transform(X_train)
Xte = baseline_vec.transform(X_test)

baseline_clf = LinearSVC(class_weight='balanced', random_state=RANDOM_SEED, max_iter=5000)
baseline_clf.fit(Xtr, y_train)
baseline_preds = baseline_clf.predict(Xte)

print(f"Baseline micro-F1: {f1_score(y_test, baseline_preds, average='macro'):.4f}")
print(classification_report(y_test, baseline_preds))

c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Baseline micro-F1: 0.6101
                   precision    recall  f1-score   support

          Billing       0.74      0.75      0.74       260
  General Inquiry       0.64      0.63      0.63       965
            Sales       0.37      0.39      0.38        66
Technical Support       0.68      0.69      0.68      1048

         accuracy                           0.66      2339
        macro avg       0.61      0.61      0.61      2339
     weighted avg       0.66      0.66      0.66      2339



##### A.2 Error Analysis — read actual misclassified examples

Rather than assume low score is a model/tuning problem, we pull actual text of misclassified tickets for most-confused class pairs.

In [3]:
results_df = pd.DataFrame({"text": X_test.values, "true": y_test.values, "pred": baseline_preds})

pairs = [('Product Support','Technical Support'), ('General Inquiry','Product Support'),
         ('Account','Technical Support'), ('General Inquiry','Technical Support')]

for true_c, pred_c in pairs:
    subset = results_df[(results_df.true == true_c) & (results_df.pred == pred_c)]
    print(f"--- TRUE={true_c} PRED={pred_c} (n={len(subset)})")
    for t in subset['text'].head(2):
        print('>', t[:220].replace(chr(10), ' '))
    print()

--- TRUE=Product Support PRED=Technical Support (n=0)

--- TRUE=General Inquiry PRED=Product Support (n=0)

--- TRUE=Account PRED=Technical Support (n=0)

--- TRUE=General Inquiry PRED=Technical Support (n=295)
> Dear Customer Support, I am writing to request enhancements to the data analytics tools that have proven to be helpful in improving our investment optimization and decision-making processes. I believe that with some upda
> Problem with Audio Input Problem with Audio Input I am encountering an audio input issue with my Rode NT-USB Mini microphone using OBS Studio version 27 on Ubuntu. I have restarted the devices, checked the connections, a



**Finding:** misclassified examples read as textbook cases of the predicted class in a common-sense reading — e.g. a dashboard bug report
labeled "Product Support" but predicted "Technical Support", or a bug
report labeled "General Inquiry". This indicates label noise in the
synthetic data for these three categories, not a fixable model problem.

**Decision:** merge `Product Support` + `Account` + `General Inquiry`
into a single `General Inquiry` category. This merge was already applied
in `01_data_cleaning.ipynb`

##### A.3 Hyperparameter tuning (on merged 4-class taxonomy)

Grid search over vocabulary size, `min_df`, class weighting, and SVM `C`.

In [4]:
tuning_results = []

for max_features in [8000, 15000, 20000, 30000]:
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1,2), min_df=1,
                          stop_words='english', sublinear_tf=True)
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    for C in [0.5, 1.0, 1.5, 2.0]:
        clf = LinearSVC(class_weight='balanced', C=C, random_state=RANDOM_SEED, max_iter=5000)
        clf.fit(Xtr, y_train)
        preds = clf.predict(Xte)
        score = f1_score(y_test, preds, average='macro')
        tuning_results.append({'max_features': max_features, 'C': C, 'macro_f1': score})
        
tuning_df = pd.DataFrame(tuning_results).sort_values('macro_f1', ascending=False)
tuning_df.head(10)

c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: Futu

,max_features,C,macro_f1
15,30000,2.0,0.665604
14,30000,1.5,0.665330
13,30000,1.0,0.655736
11,20000,2.0,0.653081
7,15000,2.0,0.652673
10,20000,1.5,0.651184
12,30000,0.5,0.650021
6,15000,1.5,0.649075
5,15000,1.0,0.647554
9,20000,1.0,0.645964


##### A.4 Compare model families with tuned vectorizer config

In [5]:
final_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=1,
                                   stop_words='english', sublinear_tf=True)
Xtr = final_vectorizer.fit_transform(X_train)
Xte = final_vectorizer.transform(X_test)

candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
    'LinearSVC': LinearSVC(class_weight='balanced', C=1.5, random_state=RANDOM_SEED, max_iter=5000),
    'MultinomialNB': MultinomialNB(),
}

intent_results = {}
intent_fitted = {}
for name, clf in candidates.items():
    clf.fit(Xtr, y_train)
    preds = clf.predict(Xte)
    score = f1_score(y_test, preds, average='macro')
    intent_results[name] = score
    intent_fitted[name] = (clf, preds)
    print(f"{name}: macro-F1 = {score:.4f}")
    
best_intent_name = max(intent_results, key=intent_results.get)
best_intent_clf, best_intent_preds = intent_fitted[best_intent_name]
print(f"\nBest: {best_intent_name} ({intent_results[best_intent_name]:.4f})")

LogisticRegression: macro-F1 = 0.5891


c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC: macro-F1 = 0.6653
MultinomialNB: macro-F1 = 0.4727

Best: LinearSVC (0.6653)


In [6]:
print(classification_report(y_test, best_intent_preds))

labels_order = sorted(df['category'].unique())
cm = confusion_matrix(y_test, best_intent_preds, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

                   precision    recall  f1-score   support

          Billing       0.82      0.75      0.78       260
  General Inquiry       0.67      0.68      0.67       965
            Sales       0.60      0.39      0.48        66
Technical Support       0.71      0.74      0.73      1048

         accuracy                           0.71      2339
        macro avg       0.70      0.64      0.67      2339
     weighted avg       0.70      0.71      0.70      2339



,Billing,General Inquiry,Sales,Technical Support
Billing,196,37,1,26
General Inquiry,27,652,7,279
Sales,5,27,26,8
Technical Support,12,252,9,775


**Result: LinearSVC, text-only TF-IDF, macro-F1 ≈ 0.67-0.70 depending on
exact train/test split** (varies slightly run-to-run because the Sales
class has only ~330 examples total — see Part C and
`training/error_analysis.md` for why small-class metrics shift with
minor data changes). This text-only config is what's used in
`training/train_intent.py` and the deployed `models/intent_model.joblib`.

**Known limitation:** Sales recall is consistently the weakest metric,
driven by sample size rather than a labeling problem — kept as its own
class since merging it away would eliminate the sales-routing use case.

#### Part B — Model B: Priority Classification

Same tuned vectorizer config as a starting point. Trained on dataset's real priority labels (LOW/MEDIUM/HIGH) — not fabricated or rule-based.

In [7]:
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    df['text'], df['priority'], test_size=0.2, random_state=RANDOM_SEED, stratify=df['priority']
)

prio_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=1,
                                  stop_words='english', sublinear_tf=True)
Xtr_p = prio_vectorizer.fit_transform(X_train_p)
Xte_p = prio_vectorizer.transform(X_test_p)

prio_candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
    'LinearSVC': LinearSVC(class_weight='balanced', C=1.5, random_state=RANDOM_SEED, max_iter=5000),
    'MultinomialNB': MultinomialNB(),
}

priority_results = {}
priority_fitted = {}
for name, clf in prio_candidates.items():
    clf.fit(Xtr_p, y_train_p)
    preds = clf.predict(Xte_p)
    score = f1_score(y_test_p, preds, average='macro')
    priority_results[name] = score
    priority_fitted[name] = (clf, preds)
    print(f"{name}: macro-F1 = {score:.4f}")
    
best_priority_name = max(priority_results, key=priority_results.get)
best_priority_clf, best_priority_preds = priority_fitted[best_priority_name]
print(f"\nBest: {best_priority_name} ({priority_results[best_priority_name]: .4f})")

LogisticRegression: macro-F1 = 0.5497


c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC: macro-F1 = 0.5968
MultinomialNB: macro-F1 = 0.3891

Best: LinearSVC ( 0.5968)


In [8]:
labels_order_p = ['LOW', 'MEDIUM', 'HIGH']
print(classification_report(y_test_p, best_priority_preds, labels=labels_order_p))

cm_p = confusion_matrix(y_test_p, best_priority_preds, labels=labels_order_p)
pd.DataFrame(cm_p, index=labels_order_p, columns=labels_order_p)

              precision    recall  f1-score   support

         LOW       0.56      0.47      0.51       461
      MEDIUM       0.63      0.62      0.63       969
        HIGH       0.62      0.69      0.65       909

    accuracy                           0.62      2339
   macro avg       0.61      0.59      0.60      2339
weighted avg       0.61      0.62      0.61      2339



,LOW,MEDIUM,HIGH
LOW,215,142,104
MEDIUM,94,604,271
HIGH,75,210,624


**Result: LinearSVC, text-only TF-IDF, macro-F1 ≈ 0.59-0.60.**

**Decision: not further tuned.** The cm shoes errors concentrated almost entirely between adjacent priority levels (LOW<->MEDIUM<->HIGH) rather than between two extremes — signature of an information ceiling (priority is partly a business
judgment call outside the text alone), not a fixable modeling problem.
This is why the deployed system blends this model's prediction with a
bounded sentiment-based adjustment instead of
tuning further. Let's verify that adjacent-confusion pattern directly:

In [9]:
extreme_confusion = cm_p[0][2] + cm_p[2][0]
total = cm_p.sum()
print(f"LOW<->HIGH direct confusion: {extreme_confusion} of {total} test rows ({extreme_confusion/total*100:.1f}%)")
print("Confirms errors are concentrated in adjacent priority levels, not random.")

LOW<->HIGH direct confusion: 179 of 2339 test rows (7.7%)
Confirms errors are concentrated in adjacent priority levels, not random.


#### Part C — Do the engioneered featured from notebook 01 actually help?

In [11]:
from scipy.sparse import hstack
from sklearn.preprocessing import MinMaxScaler

from app.features import FEATURE_NAMES


def evaluate_with_features(text_col, label_col, C_values):
    Xtr_text, Xte_text, ytr, yte = train_test_split(
        df[text_col], df[label_col], test_size=0.2, random_state=RANDOM_SEED, stratify=df[label_col] 
    )
    train_idx, test_idx = Xtr_text.index, Xte_text.index
    Xtr_tfidf = vec.fit_transform(Xtr_text)
    Xte_tfidf = vec.transform(Xte_text)
    
    # text-only baseline
    best_text_only = 0.0
    for C in C_values:
        clf = LinearSVC(class_weight='balanced', C=C, random_state=RANDOM_SEED, max_iter=5000)
        clf.fit(Xtr_tfidf, ytr)
        score = f1_score(yte, clf.predict(Xte_tfidf), average='macro')
        best_text_only = max(best_text_only, score)
    
    # with engineered features stacked
    fm = df[FEATURE_NAMES].values
    scaler = MinMaxScaler()
    Xtr_num = scaler.fit_transform(fm[df.index.isin(train_idx)])
    Xte_num = scaler.transform(fm[df.index.isin(test_idx)])
    Xtr_combined = hstack([Xtr_tfidf, Xtr_num])
    Xte_combined = hstack([Xte_tfidf, Xte_num])
    
    best_with_features = 0.0
    for C in C_values:
        clf = LinearSVC(class_weight='balanced', C=C, random_state=RANDOM_SEED, max_iter=5000)
        clf.fit(Xtr_combined, ytr)
        score = f1_score(yte, clf.predict(Xte_combined), average='macro')
        best_with_features = max(best_with_features, score)
    
    return best_text_only, best_with_features
    
C_grid = [0.5, 1.0, 1.5, 2.0, 3.0]
intent_text_only, intent_with_features = evaluate_with_features('text', 'category', C_grid)
priority_text_only, priority_with_features = evaluate_with_features('text', 'priority', C_grid)
    
print(f"Model A (intent):   text-only={intent_text_only:.4f}  with-features={intent_with_features:.4f}")
print(f"Model B (priority): text-only={priority_text_only:.4f}  with-features={priority_with_features:.4f}")
    

c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: Futu

Model A (intent):   text-only=0.6656  with-features=0.6607
Model B (priority): text-only=0.5996  with-features=0.5980
